# Tendência própria: proporções, poder por efeito e agregação em "Todos" (fase 2a, simulação)

**Ficha:** [`docs/modelagens/tendencia.md`](../../docs/modelagens/tendencia.md) · **ADR:** 0040 (proposto) · **Status:** em avaliação
· **Fase 1:** [`tendencia_01_simulacao.ipynb`](tendencia_01_simulacao.ipynb)

## 1. Pergunta

A fase 1 escolheu, por simulação, um candidato para indicadores em escala de contagem (engajamento): regressão do log do valor no
tempo com o formato como controle. Ela deixou cinco lacunas, fechadas aqui:

1. **Indicadores em proporção** (engajamento qualificado, % positivo/negativo de comentários): o modelo log-linear não se aplica.
   Qual método sustenta o contrato de quatro estados quando cada data tem um número diferente de comentários?
2. **Poder por tamanho de efeito** e **sensibilidade** a `MIN_EFEITO`, `MIN_PUBLICACOES` e `MIN_DIAS_COM_PUBLICACAO`.
3. **Visão "Todos"**: como agregar quando um perfil domina o conjunto? (A fase 1 foi inconclusiva.)
4. **Tendência não linear** (campanha, sazonalidade): o que o método linear faz?
5. **Erro de Monte Carlo** informado ao lado dos números.

## 2. Contrato dos quatro estados (igual ao da fase 1; parâmetros a calibrar)

- **Dados insuficientes**: menos de `MIN_PUBLICACOES` publicações ou de `MIN_DIAS_COM_PUBLICACAO` dias distintos. Nenhuma seta.
- **Subindo / Caindo**: o intervalo de confiança da variação no período exclui zero **e** a variação estimada passa de `MIN_EFEITO`.
- **Estável**: todo o resto (no produto: "sem evidência suficiente de mudança").

**Limite desta avaliação:** simulação mede os métodos sob as premissas do gerador, não prova que o Instagram real se comporta
assim. O dado real do piloto entra só como teste de fumaça (seção 10); a decisão final depende do *backtest* com a Coleta de 90 dias
(#256).

In [1]:
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from scipy.special import expit, logit

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
pd.set_option("display.width", 200)

PERIODO_DIAS = 90
MIN_PUBLICACOES = 10
MIN_DIAS_COM_PUBLICACAO = 8
MIN_EFEITO = 0.05
NIVEL_CONF = 0.95
N_GOVERNADORES = 26
SEED = 20261012

EFEITOS = np.array([-0.30, -0.15, 0.0, 0.15, 0.30])
PROBS = np.array([0.10, 0.15, 0.50, 0.15, 0.10])

# tamanho das simulações (dimensionado para ~15 min de execução)
N_POSTS_PROP = [20, 60, 150]
CENARIOS = ["base", "viral", "mix"]
CENARIOS_PROP = CENARIOS + ["poucos"]
REPS_PROP = 200
POR_CELULA_PODER = 1000
N_SENS = 20000
REPS_AGR = 250
N_NL = 600
T0 = time.time()

## 3. Geradores de séries

**Contagem** (copiado da fase 1, para o notebook ser autocontido): `valor = nível * (1 + efeito)^t * ruído log-normal`, com
cenários `base`, `viral` (4% de posts ×8) e `mix` (formato B rende ~3× e sua participação sobe de 20% para 60%, sem efeito real).
Estendido com `curva` (forma da tendência, para a seção 8) e `mult_nivel` (perfil dominante, seção 7).

**Proporção** (novo): cada publicação tem `n` comentários (log-normal, mediana ~40, cauda longa, variando por perfil) e `k` deles
"positivos". A proporção verdadeira parte de `p0 ~ Beta(4, 8)` (média ~0,33) e muda **linearmente na escala logit** até
`p0 * (1 + efeito)` no fim do período, então a variação verdadeira em escala log da proporção é exatamente `log(1 + efeito)`.
Há heterogeneidade entre posts além da binomial (desvio 0,4 no logit). Cenários:

- **base**: 1% de posts virais.
- **viral**: 4% de posts virais, com `n × 15` comentários e proporção deslocada (±1,2 no logit): o post polêmico que arrasta o
  peso do ajuste.
- **mix**: o formato B tem proporção maior (+0,8 no logit) e sua participação sobe de 20% para 60%, sem efeito real.
- **poucos**: poucos comentários por post (mediana ~6), onde há muitos posts com `k = 0` e a binomial é bem discreta. Testa o
  viés das aproximações (por exemplo, o log da proporção com correção de 0,5).

In [2]:
def _dias_irregulares(rng, n):
    pesos = rng.gamma(0.6, size=PERIODO_DIAS)
    return np.sort(rng.choice(PERIODO_DIAS, size=n, p=pesos / pesos.sum()))


def gerar_perfil(rng, n, efeito, cenario, curva=None, mult_nivel=1.0):
    dias = _dias_irregulares(rng, n)
    t = dias / PERIODO_DIAS
    sigma, p_viral, mult_viral = 0.5, 0.01, 5
    if cenario == "viral":
        sigma, p_viral, mult_viral = 0.6, 0.04, 8
    nivel = rng.lognormal(mean=np.log(200), sigma=0.8) * mult_nivel
    formato = np.zeros(n, dtype=int)
    if cenario == "mix":
        formato = (rng.random(n) < 0.2 + 0.4 * t).astype(int)
    tendencia = np.log1p(efeito) * t if curva is None else curva(t)
    log_y = (
        np.log(nivel)
        + tendencia
        + 1.1 * formato
        + rng.normal(0, sigma, n)
        + (rng.random(n) < p_viral) * np.log(mult_viral)
    )
    return pd.DataFrame({"dia": dias, "t": t, "formato": formato, "y": np.exp(log_y)})


def gerar_perfil_prop(rng, n, efeito, cenario, mult_com=1.0):
    dias = _dias_irregulares(rng, n)
    t = dias / PERIODO_DIAS
    sigma, p_viral = 0.4, 0.01
    if cenario == "viral":
        sigma, p_viral = 0.5, 0.04
    if cenario == "binomial_puro":  # só para a checagem de sanidade: sem sobredispersão nem posts virais
        sigma, p_viral = 0.0, 0.0
    mult_com = mult_com * (0.15 if cenario == "poucos" else 1.0)
    p0 = float(np.clip(rng.beta(4, 8), 0.03, 0.6))
    a0 = logit(p0)
    inclinacao = logit(p0 * (1 + efeito)) - a0  # logit linear em t; variação em log(p) = log(1 + efeito)
    escala = rng.lognormal(np.log(40), 0.7) * mult_com
    n_com = np.maximum(1, np.round(rng.lognormal(np.log(escala), 0.9, n))).astype(int)
    viral = rng.random(n) < p_viral
    n_com = np.where(viral, n_com * 15, n_com)
    desloc = np.where(viral, rng.choice([-1.2, 1.2], n), 0.0)
    formato = np.zeros(n, dtype=int)
    if cenario == "mix":
        formato = (rng.random(n) < 0.2 + 0.4 * t).astype(int)
    p = expit(a0 + inclinacao * t + 0.8 * formato + rng.normal(0, sigma, n) + desloc)
    k = rng.binomial(n_com, p)
    return pd.DataFrame({"dia": dias, "t": t, "formato": formato, "n": n_com, "k": k})


rng_demo = np.random.default_rng(SEED)
fig, eixos = plt.subplots(1, 3, figsize=(14, 3.4), sharey=True)
for ax, cen in zip(eixos, ["base", "mix", "poucos"]):
    d = gerar_perfil_prop(rng_demo, 100, 0.30, cen)
    ax.scatter(d["dia"], d["k"] / d["n"], s=np.clip(d["n"] / 4, 4, 120), c=d["formato"], cmap="coolwarm", alpha=0.7)
    ax.set_title(f"cenário {cen} (efeito real +30%)")
    ax.set_xlabel("dia do período")
eixos[0].set_ylabel("proporção positiva (tamanho = nº de comentários)")
plt.tight_layout()
plt.show()

## 4. Métodos comparados

**Contagem** (melhor da fase 1, reproduzido): `ols_controles`, regressão de `log(valor)` em tempo + formato, IC por t de Student.

**Proporção** (candidatos), todos devolvem a variação **relativa** da proporção no período em escala log, com IC de 95%:

| Método | Ideia |
|---|---|
| `ols_ref` | Referência da fase 1 aplicada sem adaptação: OLS de `log((k+0,5)/(n+1))` em tempo + formato, sem peso. |
| `binomial_ingenuo` | Regressão logística binomial ponderada pelo nº de comentários, com formato; IC normal, **sem** correção de sobredispersão. |
| `quase_binomial` | Igual, mas com variância escalada pela estatística de Pearson (quase-binomial) e IC por t. |
| `quase_binomial_cap` | Quase-binomial com o peso de cada post limitado a 3× a mediana dos comentários do perfil (robustez a post viral). |
| `quase_binomial_robusto` | Quase-binomial com erro-padrão robusto (sanduíche HC1), que não supõe variância binomial. |
| `cap_robusto` | Peso limitado a 3× a mediana + erro-padrão robusto. |
| `meta_regressao_logit` | Logit empírico por post, regressão ponderada com variância entre posts (efeitos aleatórios, DerSimonian-Laird) e ajuste de Knapp-Hartung. Dá a cada post um peso que satura em vez de crescer com `n`. |
| `quase_binomial_sem_controle` | Quase-binomial sem o formato (mostra o custo de não controlar o mix). |
| `janelas_wilson` | Dois blocos do período (1ª e 2ª metade), proporção agrupada de cada um com intervalo de Wilson e diferença por Newcombe; variação extrapolada para o período inteiro pela distância real entre os centros dos blocos. Não controla formato. |

A variação em escala log de uma regressão logística é `log p(1) - log p(0)`, avaliada no formato de referência, com erro-padrão
pelo método delta.

In [3]:
_T = {}


def tcrit(gl):
    gl = max(int(gl), 1)
    if gl not in _T:
        _T[gl] = stats.t.ppf(0.5 + NIVEL_CONF / 2, gl)
    return _T[gl]


Z = stats.norm.ppf(0.5 + NIVEL_CONF / 2)


def ols_rapido(y, X):
    inv = np.linalg.inv(X.T @ X)
    beta = inv @ X.T @ y
    res = y - X @ beta
    gl = len(y) - X.shape[1]
    s2 = res @ res / gl
    return beta, s2 * inv, gl


def _X(df, controles):
    cols = [np.ones(len(df)), df["t"].to_numpy()]
    if controles and df["formato"].nunique() > 1:
        cols.append(df["formato"].to_numpy())
    return np.column_stack(cols)


# ---- contagem
def _ols_ctrl_fit(df):
    b, cov, gl = ols_rapido(np.log(df["y"].to_numpy()), _X(df, True))
    return b[1], np.sqrt(cov[1, 1]), gl


def m_ols_controles(df):
    b, se, gl = _ols_ctrl_fit(df)
    c = tcrit(gl)
    return b, b - c * se, b + c * se


# ---- proporção
def logit_irls(X, k, n, maxit=40):
    y = k / n
    mu = (k + 0.5) / (n + 1)
    eta = np.log(mu / (1 - mu))
    beta = np.zeros(X.shape[1])
    for _ in range(maxit):
        mu = np.clip(expit(eta), 1e-6, 1 - 1e-6)
        w = n * mu * (1 - mu)
        z = eta + (y - mu) / (mu * (1 - mu))
        XtW = X.T * w
        novo = np.linalg.solve(XtW @ X, XtW @ z)
        eta = X @ novo
        parou = np.max(np.abs(novo - beta)) < 1e-9
        beta = novo
        if parou:
            break
    mu = np.clip(expit(eta), 1e-6, 1 - 1e-6)
    w = n * mu * (1 - mu)
    inv = np.linalg.inv((X.T * w) @ X)
    pearson = float(np.sum(n * (y - mu) ** 2 / (mu * (1 - mu))))
    return beta, inv, pearson, mu


def _delta(a, b, cov):
    # variação em escala log da proporção no período (formato de referência): log p(1) - log p(0), erro-padrão por delta
    p0, p1 = expit(a), expit(a + b)
    L = -np.log1p(np.exp(-(a + b))) + np.log1p(np.exp(-a))
    g = np.array([p0 - p1, 1 - p1])
    return L, float(np.sqrt(g @ cov[:2, :2] @ g))


def _prop_fit(t, k, n, formato=None, quase=True, robusto=False):
    cols = [np.ones(len(t)), t]
    if formato is not None and np.unique(formato).size > 1:
        cols.append(formato)
    X = np.column_stack(cols)
    beta, inv, pearson, mu = logit_irls(X, k, n)
    gl = len(t) - X.shape[1]
    if robusto:  # sanduíche (HC1): não supõe que a variância siga a binomial
        U = X * (k - n * mu)[:, None]
        cov = inv @ (U.T @ U) @ inv * len(t) / gl
    else:
        cov = (pearson / gl if quase else 1.0) * inv
    L, se = _delta(beta[0], beta[1], cov)
    crit = tcrit(gl) if (quase or robusto) else Z
    return L, se, crit


def _cap(df, mult=3.0):
    n = df["n"].to_numpy(float)
    f = np.minimum(n, mult * np.median(n)) / n
    return df["k"].to_numpy(float) * f, n * f


def _tupla(L, se, crit):
    return L, L - crit * se, L + crit * se


def m_quase(df):
    return _tupla(*_prop_fit(df["t"].to_numpy(), df["k"].to_numpy(float), df["n"].to_numpy(float), df["formato"].to_numpy()))


def m_binomial_ingenuo(df):
    return _tupla(
        *_prop_fit(df["t"].to_numpy(), df["k"].to_numpy(float), df["n"].to_numpy(float), df["formato"].to_numpy(), quase=False)
    )


def m_quase_cap(df):
    k, n = _cap(df)
    return _tupla(*_prop_fit(df["t"].to_numpy(), k, n, df["formato"].to_numpy()))


def m_quase_robusto(df):
    return _tupla(
        *_prop_fit(df["t"].to_numpy(), df["k"].to_numpy(float), df["n"].to_numpy(float), df["formato"].to_numpy(), robusto=True)
    )


def m_cap_robusto(df):
    k, n = _cap(df)
    return _tupla(*_prop_fit(df["t"].to_numpy(), k, n, df["formato"].to_numpy(), robusto=True))


def m_meta_regressao_logit(df):
    # logit empírico por post, regressão ponderada com variância entre posts (efeitos aleatórios, DerSimonian-Laird) e
    # ajuste de Knapp-Hartung para o erro-padrão
    kk = df["k"].to_numpy(float) + 0.5
    nn = df["n"].to_numpy(float) + 1.0
    z = np.log(kk / (nn - kk))
    v = 1 / kk + 1 / (nn - kk)
    X = _X(df, True)
    gl = len(z) - X.shape[1]

    def wls(w):
        XtW = X.T * w
        inv = np.linalg.inv(XtW @ X)
        return inv @ XtW @ z, inv

    w = 1 / v
    b, inv = wls(w)
    q = float(np.sum(w * (z - X @ b) ** 2))
    c = np.sum(w) - np.trace(inv @ ((X.T * w**2) @ X))
    tau2 = max(0.0, (q - gl) / c)
    w2 = 1 / (v + tau2)
    b, inv = wls(w2)
    phi = max(1.0, float(np.sum(w2 * (z - X @ b) ** 2)) / gl)
    L, se = _delta(b[0], b[1], phi * inv)
    return _tupla(L, se, tcrit(gl))


def m_quase_sem_controle(df):
    return _tupla(*_prop_fit(df["t"].to_numpy(), df["k"].to_numpy(float), df["n"].to_numpy(float), None))


def m_ols_ref(df):
    y = np.log((df["k"].to_numpy() + 0.5) / (df["n"].to_numpy() + 1))
    b, cov, gl = ols_rapido(y, _X(df, True))
    se, c = np.sqrt(cov[1, 1]), tcrit(gl)
    return b[1], b[1] - c * se, b[1] + c * se


def wilson(k, n, z=Z):
    ph = k / n
    den = 1 + z * z / n
    centro = (ph + z * z / (2 * n)) / den
    meia = z * np.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / den
    return centro - meia, centro + meia


def m_janelas_wilson(df):
    a, b = df[df["t"] < 0.5], df[df["t"] >= 0.5]
    if len(a) == 0 or len(b) == 0:
        return None
    ka, na = a["k"].sum() + 0.5, a["n"].sum() + 1.0
    kb, nb = b["k"].sum() + 0.5, b["n"].sum() + 1.0
    pa, pb = ka / na, kb / nb
    la, ua = wilson(ka, na)
    lb, ub = wilson(kb, nb)
    d = pb - pa
    lo_d = d - np.sqrt((pb - lb) ** 2 + (ua - pa) ** 2)  # Newcombe (híbrido de escore)
    hi_d = d + np.sqrt((ub - pb) ** 2 + (pa - la) ** 2)
    # extrapola para o período inteiro pela distância real entre os centros (médias de t ponderadas por comentários)
    fator = 1.0 / max(np.average(b["t"], weights=b["n"]) - np.average(a["t"], weights=a["n"]), 0.05)
    est = fator * np.log(pb / pa)
    lo = fator * np.log(max(pa + lo_d, 1e-3) / pa)
    hi = fator * np.log(max(pa + hi_d, 1e-3) / pa)
    return est, lo, hi


METODOS_PROP = {
    "ols_ref": m_ols_ref,
    "binomial_ingenuo": m_binomial_ingenuo,
    "quase_binomial": m_quase,
    "quase_binomial_cap": m_quase_cap,
    "quase_binomial_robusto": m_quase_robusto,
    "cap_robusto": m_cap_robusto,
    "meta_regressao_logit": m_meta_regressao_logit,
    "quase_binomial_sem_controle": m_quase_sem_controle,
    "janelas_wilson": m_janelas_wilson,
}


def classificar(d, min_pub=None, min_dias=None, min_efeito=None):
    min_pub = MIN_PUBLICACOES if min_pub is None else min_pub
    min_dias = MIN_DIAS_COM_PUBLICACAO if min_dias is None else min_dias
    min_efeito = MIN_EFEITO if min_efeito is None else min_efeito
    insuf = (d["npub"] < min_pub) | (d["ndias"] < min_dias) | d["est"].isna()
    est, lo, hi = np.expm1(d["est"]), np.expm1(d["lo"]), np.expm1(d["hi"])
    sobe = (lo > 0) & (est >= min_efeito)
    cai = (hi < 0) & (est <= -min_efeito)
    return pd.Series(
        np.where(insuf, "Dados insuficientes", np.where(sobe, "Subindo", np.where(cai, "Caindo", "Estável"))), index=d.index
    )


def estado_unico(npub, ndias, res, **kw):
    r = res if res is not None else (np.nan,) * 3
    d = pd.DataFrame([{"npub": npub, "ndias": ndias, "est": r[0], "lo": r[1], "hi": r[2]}])
    return classificar(d, **kw).iloc[0]

### Checagem de sanidade (com asserção)

Lição da fase 1: a primeira rodada teve um bug de deriva no ARIMA e um CUSUM sem calibração, e só foram pegos porque se conferiu
o método contra uma verdade conhecida. Aqui, antes de qualquer experimento:

1. as implementações rápidas (OLS e IRLS próprios) batem com `statsmodels` (coeficientes, e a escala de Pearson do quase-binomial);
2. em série **sem ruído** com variação conhecida de +30%, todos os métodos recuperam `log(1,3)` e o contrato diz "Subindo";
3. em série sem ruído e sem variação, o contrato diz "Estável".

In [4]:
rng_s = np.random.default_rng(1)

# 1) implementações próprias vs statsmodels
d = gerar_perfil(rng_s, 80, 0.2, "mix")
X = _X(d, True)
b_sm = sm.OLS(np.log(d["y"].to_numpy()), X).fit()
b_me, cov_me, _ = ols_rapido(np.log(d["y"].to_numpy()), X)
assert np.allclose(b_sm.params, b_me) and np.allclose(b_sm.cov_params(), cov_me)

dp = gerar_perfil_prop(rng_s, 80, 0.2, "mix")
Xp = _X(dp, True)
glm = sm.GLM(np.column_stack([dp["k"], dp["n"] - dp["k"]]), Xp, family=sm.families.Binomial()).fit()
beta_me, inv_me, pear, _ = logit_irls(Xp, dp["k"].to_numpy(float), dp["n"].to_numpy(float))
assert np.allclose(glm.params, beta_me, atol=1e-6)
assert np.isclose(glm.pearson_chi2, pear, rtol=1e-6)  # base da escala quase-binomial
assert np.allclose(glm.cov_params(), inv_me, rtol=1e-5)

# 2) série sem ruído com +30%: todo método recupera a verdade
verdade = np.log(1.3)
dc = gerar_perfil(rng_s, 250, 0.30, "base")
dc["y"] = np.exp(np.log(200) + verdade * dc["t"])
assert abs(m_ols_controles(dc)[0] - verdade) < 1e-9

dpr = gerar_perfil_prop(rng_s, 250, 0.30, "base")
p0 = 0.30
t = dpr["t"].to_numpy()
p_t = expit(logit(p0) + (logit(p0 * 1.3) - logit(p0)) * t)
dpr["n"] = np.round(rng_s.lognormal(np.log(40), 0.9, len(dpr))).clip(1) * 1.0
dpr["k"] = dpr["n"] * p_t  # esperança exata, sem ruído binomial
for nome, fn in METODOS_PROP.items():
    est = fn(dpr)[0]
    tol = 0.06 if nome == "janelas_wilson" else (0.08 if nome in ("ols_ref", "meta_regressao_logit") else 1e-6)
    print(f"{nome:30s} verdade={verdade:.3f} estimado={est:.3f}")
    assert abs(est - verdade) < tol, nome
assert estado_unico(250, 80, m_quase(dpr)) == "Subindo"

# 3) sem variação: Estável
dpr0 = dpr.copy()
dpr0["k"] = dpr0["n"] * p0
assert estado_unico(250, 80, m_quase(dpr0)) == "Estável"
# 4) no caso em que o modelo vale (binomial pura, sem sobredispersão nem post viral), o quase-binomial e o meta-regressão
#    devem ser calibrados; se não forem, o bug está no código do método, não no gerador
rng_b = np.random.default_rng(7)
cob = {"quase_binomial": [], "meta_regressao_logit": []}
fa = {"quase_binomial": [], "meta_regressao_logit": []}
for _ in range(600):
    e = rng_b.choice([0.0, 0.0, 0.3, -0.3])
    df_ = gerar_perfil_prop(rng_b, 60, e, "binomial_puro")
    for nome in cob:
        est, lo, hi = METODOS_PROP[nome](df_)
        cob[nome].append(lo <= np.log1p(e) <= hi)
        if e == 0:
            fa[nome].append(estado_unico(len(df_), df_["dia"].nunique(), (est, lo, hi)) != "Estável")
for nome in cob:
    print(f"binomial pura | {nome:22s} cobertura={np.mean(cob[nome]):.3f} falso alarme={np.mean(fa[nome]):.3f}")
    assert 0.92 <= np.mean(cob[nome]) <= 0.985 and np.mean(fa[nome]) <= 0.09, nome
print("sanidade ok")

ols_ref                        verdade=0.262 estimado=0.248
binomial_ingenuo               verdade=0.262 estimado=0.262
quase_binomial                 verdade=0.262 estimado=0.262
quase_binomial_cap             verdade=0.262 estimado=0.262
quase_binomial_robusto         verdade=0.262 estimado=0.262
cap_robusto                    verdade=0.262 estimado=0.262
meta_regressao_logit           verdade=0.262 estimado=0.256
quase_binomial_sem_controle    verdade=0.262 estimado=0.262
janelas_wilson                 verdade=0.262 estimado=0.263


binomial pura | quase_binomial         cobertura=0.955 falso alarme=0.039
binomial pura | meta_regressao_logit   cobertura=0.970 falso alarme=0.023
sanidade ok


## 5. Experimento 1: indicadores em proporção

Mesmo protocolo da fase 1: cada repetição simula uma Coleta com 26 governadores (50% sem variação, 25% de queda de 15%/30%,
25% de alta de 15%/30%). Variamos publicações por perfil (20, 60, 150) e cenário. Métricas: falso alarme, poder, direção errada,
cobertura do IC, e o **erro de Monte Carlo** (erro-padrão binomial sobre séries independentes) ao lado de cada número.

In [5]:
def simular(gerador, metodos, n_posts, cenario, reps, rng):
    linhas = []
    for _ in range(reps):
        efeitos = rng.choice(EFEITOS, size=N_GOVERNADORES, p=PROBS)
        dfs = [gerador(rng, n_posts, e, cenario) for e in efeitos]
        for nome, m in metodos.items():
            for e, df in zip(efeitos, dfs):
                try:
                    r = m(df)
                except Exception:
                    r = None
                r = r if r is not None else (np.nan,) * 3
                linhas.append((nome, n_posts, cenario, e, len(df), df["dia"].nunique(), *r))
    return pd.DataFrame(
        linhas, columns=["metodo", "n_posts", "cenario", "efeito_real", "npub", "ndias", "est", "lo", "hi"]
    )


def preparar(sim):
    sim = sim.copy()
    sim["estado"] = classificar(sim)
    sim["verdade"] = np.select([sim.efeito_real > 0, sim.efeito_real < 0], ["Subindo", "Caindo"], "Estável")
    tem = (sim.estado != "Dados insuficientes") & sim.est.notna()
    sim["cobre"] = np.where(tem, (sim.lo <= np.log1p(sim.efeito_real)) & (np.log1p(sim.efeito_real) <= sim.hi), np.nan)
    sim["erro_pp"] = np.where(tem, (np.expm1(sim.est) - sim.efeito_real).abs() * 100, np.nan)
    return sim


def _ep(p, n):
    return float(np.sqrt(max(p * (1 - p), 0) / n)) if n > 0 else np.nan


def resumo(d):
    est0, mov = d[d.verdade == "Estável"], d[d.verdade != "Estável"]
    oposto = {"Subindo": "Caindo", "Caindo": "Subindo"}
    fa = est0.estado.isin(["Subindo", "Caindo"]).mean()
    po = (mov.estado == mov.verdade).mean()
    de = (mov.estado == mov.verdade.map(oposto)).mean()
    cb = d.cobre.dropna().astype(float)
    return pd.Series(
        {
            "falso_alarme": fa, "fa_ep": _ep(fa, len(est0)),
            "poder": po, "poder_ep": _ep(po, len(mov)),
            "dir_errada": de, "dir_ep": _ep(de, len(mov)),
            "cobertura_ic": cb.mean(), "cob_ep": _ep(cb.mean(), len(cb)),
            "insuficiente": (d.estado == "Dados insuficientes").mean(),
            "erro_pp": d.erro_pp.mean(),
        }
    )


def mostrar(tab):
    # formata "valor ± erro de Monte Carlo" em pontos percentuais
    f = lambda p, e: "n/d" if pd.isna(p) else f"{100 * p:.1f} ± {100 * e:.1f}"
    saida = pd.DataFrame(index=tab.index)
    saida["falso alarme %"] = [f(p, e) for p, e in zip(tab.falso_alarme, tab.fa_ep)]
    saida["poder %"] = [f(p, e) for p, e in zip(tab.poder, tab.poder_ep)]
    saida["direção errada %"] = [f(p, e) for p, e in zip(tab.dir_errada, tab.dir_ep)]
    saida["cobertura IC %"] = [f(p, e) for p, e in zip(tab.cobertura_ic, tab.cob_ep)]
    saida["insuf. %"] = (100 * tab.insuficiente).round(1)
    saida["erro mag. (pp)"] = tab.erro_pp.round(1)
    return saida


rng = np.random.default_rng(SEED)
frames = []
for cen in CENARIOS_PROP:
    for n in N_POSTS_PROP:
        frames.append(simular(gerar_perfil_prop, METODOS_PROP, n, cen, REPS_PROP, rng))
    print(cen, "ok", round(time.time() - T0), "s")
sim_prop = preparar(pd.concat(frames, ignore_index=True))
print(len(sim_prop), "linhas;", REPS_PROP, "repetições × 26 perfis por célula")

base ok 56 s


viral ok 111 s


mix ok 168 s


poucos ok 223 s


561600 linhas; 200 repetições × 26 perfis por célula


In [6]:
print("Visão geral (todos os cenários e tamanhos), com erro de Monte Carlo (± 1 e.p.):")
geral_prop = sim_prop.groupby("metodo").apply(resumo)
mostrar(geral_prop)

Visão geral (todos os cenários e tamanhos), com erro de Monte Carlo (± 1 e.p.):


,falso alarme %,poder %,direção errada %,cobertura IC %,insuf. %,erro mag. (pp)
metodo,,,,,,
binomial_ingenuo,49.5 ± 0.3,46.0 ± 0.3,12.6 ± 0.2,50.4 ± 0.2,0.000,38.500
cap_robusto,7.1 ± 0.1,23.7 ± 0.2,0.7 ± 0.0,92.8 ± 0.1,0.000,24.800
janelas_wilson,55.1 ± 0.3,43.6 ± 0.3,16.5 ± 0.2,45.0 ± 0.2,0.000,95.600
meta_regressao_logit,4.6 ± 0.1,19.9 ± 0.2,0.4 ± 0.0,95.3 ± 0.1,0.000,18.500
ols_ref,4.9 ± 0.1,21.5 ± 0.2,0.4 ± 0.0,94.7 ± 0.1,0.000,20.000
quase_binomial,25.8 ± 0.2,30.1 ± 0.3,5.9 ± 0.1,74.2 ± 0.2,0.000,38.500
quase_binomial_cap,8.6 ± 0.2,27.0 ± 0.3,0.7 ± 0.0,91.3 ± 0.1,0.000,24.800
quase_binomial_robusto,13.8 ± 0.2,22.3 ± 0.2,2.5 ± 0.1,86.2 ± 0.1,0.000,38.500
quase_binomial_sem_controle,28.4 ± 0.3,29.8 ± 0.3,6.6 ± 0.1,71.5 ± 0.2,0.000,41.800


In [7]:
por_cen_prop = sim_prop.groupby(["cenario", "metodo"]).apply(resumo)
mostrar(por_cen_prop)

falso alarme %     poder % direção errada % cobertura IC %  insuf. %  erro mag. (pp)
cenario metodo                                                                                                          
base    binomial_ingenuo                50.7 ± 0.6  51.6 ± 0.6       10.6 ± 0.3     49.1 ± 0.4     0.000          28.800
        cap_robusto                      7.1 ± 0.3  28.1 ± 0.5        0.5 ± 0.1     92.8 ± 0.2     0.000          18.000
        janelas_wilson                  52.6 ± 0.6  48.2 ± 0.6       13.3 ± 0.4     47.6 ± 0.4     0.000          38.000
        meta_regressao_logit             4.3 ± 0.2  24.5 ± 0.5        0.2 ± 0.0     95.6 ± 0.2     0.000          15.600
        ols_ref                          5.0 ± 0.2  26.1 ± 0.5        0.1 ± 0.0     94.8 ± 0.2     0.000          17.900
        quase_binomial                  24.8 ± 0.5  33.2 ± 0.5        4.6 ± 0.2     75.3 ± 0.3     0.000          28.800
        quase_binomial_cap               8.9 ± 0.3  31.3 ± 0.5        0.6 ± 0.1     91.1 ± 0.2     0.000          18.000
        quase_binomial_robusto          12.1 ± 0.4  24.9 ± 0.5        1.7 ± 0.1     87.7 ± 0.3     0.000          28.800
        quase_binomial_sem_controle     24.8 ± 0.5  33.2 ± 0.5        4.6 ± 0.2     75.3 ± 0.3     0.000          28.800
mix     binomial_ingenuo                50.2 ± 0.6  51.7 ± 0.6       10.3 ± 0.3     50.1 ± 0.4     0.000          28.100
        cap_robusto                      7.4 ± 0.3  28.1 ± 0.5        0.6 ± 0.1     92.6 ± 0.2     0.000          18.700
        janelas_wilson                  66.5 ± 0.5  49.0 ± 0.6       19.6 ± 0.4     33.6 ± 0.4     0.000          59.800
        meta_regressao_logit             4.2 ± 0.2  24.3 ± 0.5        0.3 ± 0.1     95.7 ± 0.2     0.000          16.000
        ols_ref                          5.2 ± 0.3  25.4 ± 0.5        0.3 ± 0.1     94.5 ± 0.2     0.000          16.900
        quase_binomial                  23.5 ± 0.5  32.6 ± 0.5        4.2 ± 0.2     76.8 ± 0.3     0.000          28.100
        quase_binomial_cap               8.7 ± 0.3  31.8 ± 0.5        0.6 ± 0.1     91.2 ± 0.2     0.000          18.700
        quase_binomial_robusto          13.3 ± 0.4  25.8 ± 0.5        1.9 ± 0.2     86.8 ± 0.3     0.000          28.100
        quase_binomial_sem_controle     33.8 ± 0.5  31.3 ± 0.5        7.2 ± 0.3     66.3 ± 0.4     0.000          41.200
poucos  binomial_ingenuo                26.4 ± 0.5  29.1 ± 0.5        5.7 ± 0.3     74.2 ± 0.4     0.000          42.400
        cap_robusto                      6.2 ± 0.3  17.3 ± 0.4        0.8 ± 0.1     93.6 ± 0.2     0.000          38.500
        janelas_wilson                  27.8 ± 0.5  27.4 ± 0.5        6.8 ± 0.3     72.6 ± 0.4     0.000          68.400
        meta_regressao_logit             6.6 ± 0.3  14.7 ± 0.4        0.9 ± 0.1     93.2 ± 0.2     0.000          23.200
        ols_ref                          5.1 ± 0.2  12.8 ± 0.4        0.6 ± 0.1     94.5 ± 0.2     0.000          24.500
        quase_binomial                  17.3 ± 0.4  22.4 ± 0.5        3.5 ± 0.2     83.1 ± 0.3     0.000          42.400
        quase_binomial_cap               5.8 ± 0.3  17.6 ± 0.4        0.5 ± 0.1     94.0 ± 0.2     0.000          38.500
        quase_binomial_robusto          10.8 ± 0.3  18.4 ± 0.4        2.1 ± 0.2     89.3 ± 0.2     0.000          42.400
        quase_binomial_sem_controle     17.3 ± 0.4  22.4 ± 0.5        3.5 ± 0.2     83.1 ± 0.3     0.000          42.400
viral   binomial_ingenuo                71.0 ± 0.5  51.2 ± 0.6       23.6 ± 0.5     28.3 ± 0.4     0.000          54.600
        cap_robusto                      7.7 ± 0.3  21.1 ± 0.5        1.0 ± 0.1     92.3 ± 0.2     0.000          24.000
        janelas_wilson                  73.7 ± 0.5  49.3 ± 0.6       26.1 ± 0.5     26.2 ± 0.4     0.000         216.300
        meta_regressao_logit             3.1 ± 0.2  15.9 ± 0.4        0.4 ± 0.1     96.7 ± 0.1     0.000          19.200
        ols_ref                          4.

In [8]:
por_n_prop = sim_prop.groupby(["n_posts", "metodo"]).apply(resumo)
fig, eixos = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, col, titulo in zip(
    eixos, ["falso_alarme", "poder", "cobertura_ic"], ["Falso alarme (menor é melhor)", "Poder (maior é melhor)", "Cobertura do IC (alvo 0,95)"]
):
    por_n_prop[col].unstack("metodo").plot(ax=ax, marker="o")
    ax.set_title(titulo)
    ax.set_xlabel("publicações por perfil")
    ax.set_xscale("log")
    ax.set_xticks(N_POSTS_PROP)
    ax.set_xticklabels(N_POSTS_PROP)
    if col == "cobertura_ic":
        ax.axhline(0.95, color="gray", ls="--")
    ax.legend(fontsize=6)
plt.tight_layout()
plt.show()
mostrar(por_n_prop)

falso alarme %     poder % direção errada % cobertura IC %  insuf. %  erro mag. (pp)
n_posts metodo                                                                                                          
20      binomial_ingenuo                38.3 ± 0.5  32.9 ± 0.5       10.5 ± 0.3     61.9 ± 0.3     0.000          53.200
        cap_robusto                      8.9 ± 0.3  13.0 ± 0.3        1.4 ± 0.1     91.0 ± 0.2     0.000          42.900
        janelas_wilson                  43.3 ± 0.5  33.2 ± 0.5       13.7 ± 0.3     56.8 ± 0.3     0.000         197.700
        meta_regressao_logit             4.5 ± 0.2   7.9 ± 0.3        0.6 ± 0.1     95.5 ± 0.1     0.000          28.900
        ols_ref                          4.7 ± 0.2   8.6 ± 0.3        0.6 ± 0.1     95.0 ± 0.2     0.000          32.000
        quase_binomial                  14.8 ± 0.3  15.9 ± 0.4        3.7 ± 0.2     85.2 ± 0.2     0.000          53.200
        quase_binomial_cap               7.9 ± 0.3  12.3 ± 0.3        1.1 ± 0.1     92.1 ± 0.2     0.000          42.900
        quase_binomial_robusto          15.2 ± 0.4  16.0 ± 0.4        3.7 ± 0.2     84.8 ± 0.2     0.000          53.200
        quase_binomial_sem_controle     16.5 ± 0.4  16.0 ± 0.4        4.2 ± 0.2     83.7 ± 0.3     0.000          57.500
60      binomial_ingenuo                50.5 ± 0.5  47.9 ± 0.5       13.4 ± 0.3     49.4 ± 0.3     0.000          35.100
        cap_robusto                      6.6 ± 0.2  21.8 ± 0.4        0.5 ± 0.1     93.3 ± 0.2     0.000          19.500
        janelas_wilson                  56.4 ± 0.5  45.5 ± 0.5       17.1 ± 0.4     43.9 ± 0.3     0.000          51.900
        meta_regressao_logit             4.6 ± 0.2  18.5 ± 0.4        0.4 ± 0.1     95.2 ± 0.1     0.000          16.200
        ols_ref                          5.3 ± 0.2  19.3 ± 0.4        0.3 ± 0.1     94.6 ± 0.2     0.000          17.300
        quase_binomial                  26.2 ± 0.4  31.0 ± 0.5        6.4 ± 0.2     73.7 ± 0.3     0.000          35.100
        quase_binomial_cap               8.6 ± 0.3  25.6 ± 0.4        0.8 ± 0.1     91.2 ± 0.2     0.000          19.500
        quase_binomial_robusto          14.2 ± 0.3  23.1 ± 0.4        2.6 ± 0.2     85.8 ± 0.2     0.000          35.100
        quase_binomial_sem_controle     28.8 ± 0.4  31.0 ± 0.5        7.1 ± 0.3     71.3 ± 0.3     0.000          37.900
150     binomial_ingenuo                59.7 ± 0.5  57.2 ± 0.5       13.9 ± 0.3     40.0 ± 0.3     0.000          27.200
        cap_robusto                      5.7 ± 0.2  36.3 ± 0.5        0.2 ± 0.0     94.2 ± 0.2     0.000          12.000
        janelas_wilson                  65.5 ± 0.5  52.0 ± 0.5       18.6 ± 0.4     34.3 ± 0.3     0.000          37.200
        meta_regressao_logit             4.6 ± 0.2  33.3 ± 0.5        0.3 ± 0.1     95.1 ± 0.1     0.000          10.400
        ols_ref                          4.8 ± 0.2  36.8 ± 0.5        0.2 ± 0.0     94.6 ± 0.2     0.000          10.800
        quase_binomial                  36.5 ± 0.5  43.3 ± 0.5        7.5 ± 0.3     63.6 ± 0.3     0.000          27.200
        quase_binomial_cap               9.2 ± 0.3  43.1 ± 0.5        0.3 ± 0.1     90.7 ± 0.2     0.000          12.000
        quase_binomial_robusto          11.9 ± 0.3  27.7 ± 0.4        1.4 ± 0.1     88.1 ± 0.2     0.000          27.200
        quase_binomial_sem_controle     39.9 ± 0.5  42.3 ± 0.5        8.6 ± 0.3     59.7 ± 0.3     0.000          29.800

### Regra de escolha do candidato de proporção

Regra aplicada em código logo abaixo, e o resultado alimenta os experimentos seguintes: só entra quem for calibrado nos quatro
cenários, isto é, falso alarme máximo ≤ 6% e cobertura mínima do IC ≥ 93% (a nominal é 95%). Entre os que passam, vence o de maior
poder médio; empate técnico (diferença menor que 2 erros-padrão) é resolvido pelo método mais simples da lista.

**Registro de transparência:** a primeira versão da regra usava limites mais frouxos (falso alarme ≤ 8%, cobertura ≥ 92%). Com
eles, `cap_robusto` vencia, mas passava raspando (falso alarme máximo 7,7% e cobertura mínima 92,3%, com erro de Monte Carlo de
~0,3 pp) e tinha o maior erro de magnitude entre os calibrados. Como o objetivo é um método com folga de calibração, e não o que
passa no limite, os limites foram endurecidos para 6% e 93% depois de ver essa primeira rodada. A mudança foi motivada pelo
resultado, então vale como julgamento, não como regra cega: a tabela abaixo mostra todos os métodos para quem preferir outro corte.

In [9]:
FA_MAX, COB_MIN = 0.06, 0.93
por_cen = por_cen_prop.reset_index()
agg = por_cen.groupby("metodo").agg(fa_max=("falso_alarme", "max"), cob_min=("cobertura_ic", "min"), poder_medio=("poder", "mean"), poder_ep=("poder_ep", "mean"))
agg["calibrado"] = (agg.fa_max <= FA_MAX) & (agg.cob_min >= COB_MIN)
print("Calibração pior caso entre cenários (limites: falso alarme ≤ 6%, cobertura ≥ 93%):")
display(agg.round(3).sort_values("poder_medio", ascending=False))
elegiveis = agg[agg.calibrado].sort_values("poder_medio", ascending=False)
assert len(elegiveis) > 0, "nenhum candidato de proporção calibrado: investigar métodos antes de seguir"
ordem_simplicidade = list(METODOS_PROP)  # ordem da lista = do mais simples ao mais elaborado
topo = elegiveis.iloc[0]
empatados = elegiveis[elegiveis.poder_medio >= topo.poder_medio - 2 * topo.poder_ep]
PROP_CANDIDATO = min(empatados.index, key=ordem_simplicidade.index)
print("Candidato de proporção escolhido pela regra:", PROP_CANDIDATO, "| empatados tecnicamente:", list(empatados.index))

Calibração pior caso entre cenários (limites: falso alarme ≤ 6%, cobertura ≥ 93%):


,fa_max,cob_min,poder_medio,poder_ep,calibrado
metodo,,,,,
binomial_ingenuo,0.710,0.283,0.459,0.006,False
janelas_wilson,0.737,0.262,0.435,0.006,False
quase_binomial,0.379,0.615,0.300,0.005,False
quase_binomial_sem_controle,0.379,0.615,0.297,0.005,False
quase_binomial_cap,0.109,0.889,0.269,0.005,False
cap_robusto,0.077,0.923,0.236,0.005,False
quase_binomial_robusto,0.189,0.811,0.222,0.005,False
ols_ref,0.052,0.945,0.215,0.005,True
meta_regressao_logit,0.066,0.932,0.199,0.004,False


Candidato de proporção escolhido pela regra: ols_ref | empatados tecnicamente: ['ols_ref']


### Reprodução da fase 1 para contagem (referência cruzada)

Para garantir que o gerador de contagem copiado e o OLS com controles continuam calibrados, repete-se o essencial da fase 1 (menos
repetições). Serve de referência para o poder por efeito e como controle do restante do notebook.

In [10]:
rng = np.random.default_rng(SEED + 3)
fr = []
for cen in CENARIOS:
    for n in [40, 100]:
        fr.append(simular(gerar_perfil, {"ols_controles": m_ols_controles}, n, cen, max(REPS_PROP // 2, 2), rng))
ref_cont = preparar(pd.concat(fr, ignore_index=True))
print("OLS com controles, contagem (referência da fase 1):")
mostrar(ref_cont.groupby("cenario").apply(resumo))

OLS com controles, contagem (referência da fase 1):


,falso alarme %,poder %,direção errada %,cobertura IC %,insuf. %,erro mag. (pp)
cenario,,,,,,
base,5.1 ± 0.4,16.1 ± 0.7,0.2 ± 0.1,95.2 ± 0.3,0.000,19.800
mix,4.8 ± 0.4,17.6 ± 0.8,0.5 ± 0.1,95.0 ± 0.3,0.000,20.800
viral,5.6 ± 0.5,10.6 ± 0.6,0.5 ± 0.1,94.6 ± 0.3,0.000,28.700


## 6. Experimento 2: poder por tamanho de efeito e sensibilidade aos limiares

Compara o melhor método da fase 1 (`ols_controles`, contagem) com o melhor candidato de proporção do experimento 1 (variável `PROP_CANDIDATO`; a justificativa da escolha está na seção de conclusões).

**6a. Poder por efeito.** Efeito fixo por célula (de -50% a +50%), 4 tamanhos de série, cenário `base`. "Poder" = fração de séries
classificadas na direção certa. "Efeito mínimo detectável a 80%" (EMD80) = menor efeito com poder ≥ 80% (interpolado).

In [11]:
GRADE_EFEITOS = np.array([-0.5, -0.3, -0.2, -0.1, -0.05, 0.0, 0.05, 0.1, 0.2, 0.3, 0.5])
N_PODER = [20, 50, 100, 250]


def poder_por_efeito(gerador, metodo, rng, cenario="base"):
    linhas = []
    for n in N_PODER:
        for e in GRADE_EFEITOS:
            for _ in range(POR_CELULA_PODER):
                df = gerador(rng, n, e, cenario)
                try:
                    r = metodo(df)
                except Exception:
                    r = (np.nan,) * 3
                linhas.append((n, e, len(df), df["dia"].nunique(), *r))
    d = pd.DataFrame(linhas, columns=["n_posts", "efeito", "npub", "ndias", "est", "lo", "hi"])
    d["estado"] = classificar(d)
    return d


def curva_poder(d):
    certo = np.where(d.efeito > 0, "Subindo", np.where(d.efeito < 0, "Caindo", "Estável"))
    oposto = np.where(d.efeito > 0, "Caindo", np.where(d.efeito < 0, "Subindo", "x"))
    d = d.assign(acerto=(d.estado == certo), erro_dir=(d.estado == oposto), alarme=d.estado.isin(["Subindo", "Caindo"]))
    g = d.groupby(["n_posts", "efeito"])
    saida = pd.DataFrame({"poder": g.acerto.mean(), "dir_errada": g.erro_dir.mean(), "alarme": g.alarme.mean()})
    zero = saida.index.get_level_values("efeito") == 0
    saida["poder"] = np.where(zero, saida["alarme"], saida["poder"])  # no efeito 0, a coluna é a % de seta indevida
    saida["ep"] = np.sqrt(saida.poder * (1 - saida.poder) / g.size())
    return saida


def emd80(c, n, sinal):
    s = c.xs(n, level="n_posts")
    s = s[(s.index * sinal) > 0].sort_index(ascending=(sinal > 0))
    xs, ys = np.abs(s.index.to_numpy()), s["poder"].to_numpy()
    for i in range(1, len(xs)):
        if ys[i] >= 0.8 > ys[i - 1]:
            return float(xs[i - 1] + (0.8 - ys[i - 1]) * (xs[i] - xs[i - 1]) / (ys[i] - ys[i - 1]))
    return float(xs[0]) if ys[0] >= 0.8 else np.nan


rng = np.random.default_rng(SEED + 11)
pw_cont = poder_por_efeito(gerar_perfil, m_ols_controles, rng)
print("contagem ok", round(time.time() - T0), "s")
pw_prop = poder_por_efeito(gerar_perfil_prop, METODOS_PROP[PROP_CANDIDATO], rng)
print("proporção ok", round(time.time() - T0), "s")
c_cont, c_prop = curva_poder(pw_cont), curva_poder(pw_prop)

fig, eixos = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, c, titulo in zip(eixos, [c_cont, c_prop], ["Contagem: ols_controles", f"Proporção: {PROP_CANDIDATO}"]):
    for n in N_PODER:
        s = c.xs(n, level="n_posts")
        ax.plot(100 * s.index, 100 * s.poder, marker="o", label=f"{n} publicações")
    ax.axhline(80, color="gray", ls="--", lw=0.8)
    ax.axhline(5, color="gray", ls=":", lw=0.8)
    ax.set_title(titulo + " (em 0 = falso alarme)")
    ax.set_xlabel("variação real no período (%)")
eixos[0].set_ylabel("% classificado corretamente (efeito 0: % de seta indevida)")
eixos[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

emd = pd.DataFrame(
    {
        (nome, lado): [emd80(c, n, s) * 100 for n in N_PODER]
        for nome, c in [("contagem", c_cont), ("proporção", c_prop)]
        for lado, s in [("alta", 1), ("queda", -1)]
    },
    index=pd.Index(N_PODER, name="publicações"),
)
print("Efeito mínimo detectável a 80% (em %; NaN = não atinge 80% até ±50%):")
emd

contagem ok 254 s


proporção ok 282 s
Efeito mínimo detectável a 80% (em %; NaN = não atinge 80% até ±50%):


contagem        proporção       
                alta  queda      alta  queda
publicações                                 
20               NaN    NaN       NaN    NaN
50               NaN    NaN       NaN    NaN
100              NaN 43.205    47.852 43.039
250           42.919 28.828    29.302 26.498

In [12]:
def tabela_poder(c, titulo):
    t = (100 * c.poder).unstack("n_posts").round(0)
    ep = (100 * c.ep).unstack("n_posts").max().max()
    print(f"{titulo}: poder % por efeito (linhas) e publicações (colunas); erro de MC máx ≈ {ep:.1f} pp")
    return t


display(tabela_poder(c_cont, "Contagem, ols_controles"))
display(tabela_poder(c_prop, f"Proporção, {PROP_CANDIDATO}"))

Contagem, ols_controles: poder % por efeito (linhas) e publicações (colunas); erro de MC máx ≈ 1.6 pp


n_posts,20,50,100,250
efeito,,,,
-0.500,35.000,73.000,96.000,100.000
-0.300,12.000,28.000,49.000,84.000
-0.200,6.000,13.000,23.000,46.000
-0.100,4.000,6.000,10.000,13.000
-0.050,4.000,4.000,4.000,7.000
0.000,4.000,6.000,5.000,6.000
0.050,3.000,4.000,5.000,5.000
0.100,4.000,6.000,6.000,12.000
0.200,6.000,9.000,18.000,36.000


Proporção, ols_ref: poder % por efeito (linhas) e publicações (colunas); erro de MC máx ≈ 1.6 pp


n_posts,20,50,100,250
efeito,,,,
-0.500,41.000,78.000,93.000,99.000
-0.300,18.000,36.000,56.000,89.000
-0.200,10.000,18.000,33.000,63.000
-0.100,4.000,8.000,11.000,24.000
-0.050,3.000,3.000,6.000,10.000
0.000,4.000,5.000,5.000,5.000
0.050,2.000,5.000,5.000,8.000
0.100,4.000,6.000,12.000,22.000
0.200,8.000,19.000,33.000,56.000


**6b. Sensibilidade aos limiares.** Séries com número de publicações heterogêneo entre perfis (log-normal, mediana 30, de 3 a 300),
efeitos sorteados como no experimento 1, cenário `base`. Guardam-se estimativa e IC e **reaplica-se o contrato** com limiares
diferentes sem regenerar nada. Poder = fração das séries com variação real (±15%/±30%) classificada na direção certa (insuficiente
conta como erro); "poder entre classificadas" exclui as de dados insuficientes.

In [13]:
def sensibilidade_base(gerador, metodo, rng, n_series):
    linhas = []
    for _ in range(n_series):
        e = rng.choice(EFEITOS, p=PROBS)
        n = int(np.clip(np.round(rng.lognormal(np.log(30), 0.9)), 3, 300))
        df = gerador(rng, n, e, "base")
        try:
            r = metodo(df)
        except Exception:
            r = (np.nan,) * 3
        linhas.append((e, len(df), df["dia"].nunique(), *r))
    d = pd.DataFrame(linhas, columns=["efeito_real", "npub", "ndias", "est", "lo", "hi"])
    d["verdade"] = np.select([d.efeito_real > 0, d.efeito_real < 0], ["Subindo", "Caindo"], "Estável")
    return d


def metricas_limiar(d, **kw):
    est = classificar(d, **kw)
    est0, mov = d.verdade == "Estável", d.verdade != "Estável"
    cls = est != "Dados insuficientes"
    fa = est[est0].isin(["Subindo", "Caindo"]).mean()
    po = (est[mov] == d.verdade[mov]).mean()
    return {
        "insuf %": 100 * (~cls).mean(),
        "falso alarme %": 100 * fa,
        "poder %": 100 * po,
        "poder classif. %": 100 * (est[mov & cls] == d.verdade[mov & cls]).mean(),
        "dir. errada %": 100 * (est[mov & cls] == d.verdade[mov & cls].map({"Subindo": "Caindo", "Caindo": "Subindo"})).mean(),
        "ep poder": 100 * _ep(po, mov.sum()),
    }


rng = np.random.default_rng(SEED + 21)
sens = {
    "contagem": sensibilidade_base(gerar_perfil, m_ols_controles, rng, N_SENS),
    "proporção": sensibilidade_base(gerar_perfil_prop, METODOS_PROP[PROP_CANDIDATO], rng, N_SENS),
}
print("n series:", N_SENS, "por indicador;", round(time.time() - T0), "s")

MEFS = [0.0, 0.02, 0.05, 0.10, 0.15, 0.20]
linhas = []
for nome, d in sens.items():
    for me in MEFS:
        linhas.append({"indicador": nome, "MIN_EFEITO": me, **metricas_limiar(d, min_efeito=me)})
sens_mef = pd.DataFrame(linhas).set_index(["indicador", "MIN_EFEITO"])
print("Sensibilidade a MIN_EFEITO (MIN_PUBLICACOES=10, MIN_DIAS=8):")
sens_mef.round(1)

n series: 20000 por indicador; 305 s
Sensibilidade a MIN_EFEITO (MIN_PUBLICACOES=10, MIN_DIAS=8):


insuf %  falso alarme %  poder %  poder classif. %  dir. errada %  ep poder
indicador MIN_EFEITO                                                                             
contagem  0.000        10.800           4.200   11.800            13.200          0.500     0.300
          0.020        10.800           4.200   11.800            13.200          0.500     0.300
          0.050        10.800           4.200   11.800            13.200          0.500     0.300
          0.100        10.800           4.200   11.800            13.200          0.500     0.300
          0.150        10.800           4.200   11.800            13.200          0.500     0.300
          0.200        10.800           4.200   11.800            13.200          0.500     0.300
proporção 0.000         9.700           4.100   17.500            19.400          0.500     0.400
          0.020         9.700           4.100   17.500            19.400          0.500     0.400
          0.050         9.700           4.100   17.500            19.400          0.500     0.400
          0.100         9.700           4.100   17.500            19.400          0.500     0.400
          0.150         9.700           4.100   17.500            19.400          0.500     0.400
          0.200         9.700           3.900   17.000            18.800          0.500     0.400

In [14]:
PARES = [(5, 4), (10, 8), (15, 10), (20, 15), (30, 20), (50, 30)]
linhas = []
for nome, d in sens.items():
    for mp, md_ in PARES:
        linhas.append(
            {"indicador": nome, "MIN_PUB": mp, "MIN_DIAS": md_, **metricas_limiar(d, min_pub=mp, min_dias=md_)}
        )
sens_min = pd.DataFrame(linhas).set_index(["indicador", "MIN_PUB", "MIN_DIAS"])
print("Sensibilidade aos mínimos de publicações/dias (MIN_EFEITO=5%):")
sens_min.round(1)

Sensibilidade aos mínimos de publicações/dias (MIN_EFEITO=5%):


insuf %  falso alarme %  poder %  poder classif. %  dir. errada %  ep poder
indicador MIN_PUB MIN_DIAS                                                                             
contagem  5       4           2.000           4.600   12.100            12.400          0.500     0.300
          10      8          10.800           4.200   11.800            13.200          0.500     0.300
          15      10         21.200           3.700   11.200            14.200          0.500     0.300
          20      15         32.700           3.100   10.400            15.500          0.400     0.300
          30      20         50.000           2.400    9.000            18.100          0.400     0.300
          50      30         73.100           1.100    6.200            23.000          0.300     0.200
proporção 5       4           1.500           4.500   18.000            18.300          0.500     0.400
          10      8           9.700           4.100   17.500            19.400          0.500     0.400
          15      10         20.400           3.600   16.600            20.800          0.400     0.400
          20      15         32.300           3.100   15.400            22.700          0.300     0.400
          30      20         50.200           2.300   13.400            26.600          0.300     0.300
          50      30         73.200           1.100    8.900            32.500          0.200     0.300

## 7. Experimento 3: agregação da visão "Todos" com dominância forte

Decisão de produto já tomada: em "Todos" a tendência é a do **conjunto**, e o conjunto deve representar o grupo (um perfil que
publica muito não pode dominá-lo). Portanto a **verdade** do conjunto é a média (log) dos efeitos dos 26 perfis com peso igual.

Um perfil dominante (índice 0) tem nível **×D** (contagem) ou **×D comentários por post** (proporção) e publica 3× mais. Variamos
`D` em 1 (só o volume de posts), 10, 30 e 100. Três cenários de efeito:

- **A**: o dominante cai 30%, os outros 25 ficam estáveis. Verdade do conjunto ≈ -1,4%: "Estável".
- **B**: os outros 25 sobem 15%, o dominante fica estável. Verdade ≈ +14,4%: "Subindo".
- **C**: os outros 25 sobem 15%, o dominante **cai** 30%. Verdade ≈ +12,8%: "Subindo". Soma bruta deve enxergar "Caindo".

Cinco formas de agregar: `soma_bruta` (soma por dia, perfil dominante manda), `normalizada` (cada perfil dividido pelo próprio
nível, ou, em proporção, massa de comentários igualada entre perfis, e todos os posts juntos; **o perfil que publica mais ainda
pesa mais em contagem**), `normalizada_peso_igual` (só contagem: igual à anterior, mas cada post pesa 1/n do seu perfil, então cada
perfil pesa igual), `mediana_tendencias` (mediana das variações
individuais, IC por *bootstrap* de perfis) e `meta_efeitos_aleatorios` (média ponderada das variações individuais, com variância
entre perfis estimada por DerSimonian–Laird). Métricas: acerto do estado vs. verdade e direção errada (Subindo no lugar de Caindo
ou o inverso), com erro de Monte Carlo. Contagem usa `ols` em log; proporção, quase-binomial.

In [15]:
def efeitos_agregacao(cen):
    ef = np.zeros(N_GOVERNADORES)
    if cen == "A":
        ef[0] = -0.30
    elif cen == "B":
        ef[1:] = 0.15
    elif cen == "C":
        ef[1:] = 0.15
        ef[0] = -0.30
    return ef


def meta_re(b, se):
    w = 1 / se**2
    mu_f = np.sum(w * b) / np.sum(w)
    q = np.sum(w * (b - mu_f) ** 2)
    c = np.sum(w) - np.sum(w**2) / np.sum(w)
    tau2 = max(0.0, (q - (len(b) - 1)) / c)
    w2 = 1 / (se**2 + tau2)
    mu = np.sum(w2 * b) / np.sum(w2)
    ep = np.sqrt(1 / np.sum(w2))
    return mu, mu - Z * ep, mu + Z * ep


def mediana_boot(b, rng, B=400):
    idx = rng.integers(0, len(b), size=(B, len(b)))
    meds = np.median(b[idx], axis=1)
    lo, hi = np.quantile(meds, [(1 - NIVEL_CONF) / 2, 1 - (1 - NIVEL_CONF) / 2])
    return float(np.median(b)), float(lo), float(hi)


def agregar_contagem(dfs, rng):
    todos = pd.concat(dfs)
    d1 = todos.groupby("dia", as_index=False).agg(y=("y", "sum"))
    b1, cov1, gl1 = ols_rapido(np.log(d1["y"].to_numpy()), np.column_stack([np.ones(len(d1)), d1["dia"] / PERIODO_DIAS]))
    c1 = tcrit(gl1)
    soma = (b1[1], b1[1] - c1 * np.sqrt(cov1[1, 1]), b1[1] + c1 * np.sqrt(cov1[1, 1]))
    norm = pd.concat([x.assign(y=x["y"] / x["y"].mean()) for x in dfs])
    b2, cov2, gl2 = ols_rapido(np.log(norm["y"].to_numpy()), np.column_stack([np.ones(len(norm)), norm["t"]]))
    c2 = tcrit(gl2)
    normalizada = (b2[1], b2[1] - c2 * np.sqrt(cov2[1, 1]), b2[1] + c2 * np.sqrt(cov2[1, 1]))
    # normalizada + peso igual por perfil (cada post pesa 1/n do perfil, então o perfil que publica mais não pesa mais)
    nw = pd.concat([x.assign(y=x["y"] / x["y"].mean(), w=1.0 / len(x)) for x in dfs])
    sw = np.sqrt(nw["w"].to_numpy() / nw["w"].mean())
    Xw = np.column_stack([np.ones(len(nw)), nw["t"]]) * sw[:, None]
    b3, cov3, gl3 = ols_rapido(np.log(nw["y"].to_numpy()) * sw, Xw)
    c3 = tcrit(gl3)
    normalizada_pi = (b3[1], b3[1] - c3 * np.sqrt(cov3[1, 1]), b3[1] + c3 * np.sqrt(cov3[1, 1]))
    ajustes = np.array([_ols_ctrl_fit(x)[:2] for x in dfs])
    return {
        "soma_bruta": soma,
        "normalizada": normalizada,
        "normalizada_peso_igual": normalizada_pi,
        "mediana_tendencias": mediana_boot(ajustes[:, 0], rng),
        "meta_efeitos_aleatorios": meta_re(ajustes[:, 0], ajustes[:, 1]),
    }


def _se_de(r):
    return (r[2] - r[1]) / (2 * Z)


def agregar_proporcao(dfs, rng):
    metodo = METODOS_PROP[PROP_CANDIDATO]
    todos = pd.concat(dfs)
    d1 = todos.groupby("dia", as_index=False).agg(k=("k", "sum"), n=("n", "sum"))
    d1["t"] = d1["dia"] / PERIODO_DIAS
    d1["formato"] = 0
    soma = metodo(d1)
    alvo = np.mean([x["n"].sum() for x in dfs])
    norm = pd.concat([x.assign(k=x["k"] * alvo / x["n"].sum(), n=x["n"] * alvo / x["n"].sum()) for x in dfs])
    normalizada = metodo(norm)
    rs = [metodo(x) for x in dfs]
    aj = np.array([[r[0], _se_de(r)] for r in rs])
    return {
        "soma_bruta": soma,
        "normalizada": normalizada,
        "mediana_tendencias": mediana_boot(aj[:, 0], rng),
        "meta_efeitos_aleatorios": meta_re(aj[:, 0], aj[:, 1]),
    }


def rodar_agregacao(tipo, cen, D, reps, rng, n=100):
    ef = efeitos_agregacao(cen)
    verdade_log = float(np.mean(np.log1p(ef)))
    verdade_estado = estado_unico(10**6, 10**6, (verdade_log, verdade_log, verdade_log))
    linhas = []
    for _ in range(reps):
        if tipo == "contagem":
            dfs = [
                gerar_perfil(rng, n * (3 if i == 0 else 1), ef[i], "base", mult_nivel=(D if i == 0 else 1.0))
                for i in range(N_GOVERNADORES)
            ]
            res = agregar_contagem(dfs, rng)
        else:
            dfs = [
                gerar_perfil_prop(rng, n * (3 if i == 0 else 1), ef[i], "base", mult_com=(D if i == 0 else 1.0))
                for i in range(N_GOVERNADORES)
            ]
            res = agregar_proporcao(dfs, rng)
        for modo, r in res.items():
            linhas.append(
                {"tipo": tipo, "cenario": cen, "D": D, "modo": modo, "est": r[0], "lo": r[1], "hi": r[2],
                 "verdade_estado": verdade_estado, "verdade_log": verdade_log, "npub": 10**6, "ndias": 10**6}
            )
    d = pd.DataFrame(linhas)
    d["estado"] = classificar(d)
    return d


rng = np.random.default_rng(SEED + 31)
frames = []
for tipo in ["contagem", "proporção"]:
    for cen in ["A", "B", "C"]:
        for D in [1, 10, 30, 100]:
            frames.append(rodar_agregacao(tipo, cen, D, REPS_AGR, rng))
    print(tipo, "ok", round(time.time() - T0), "s")
agr = pd.concat(frames, ignore_index=True)
oposto = {"Subindo": "Caindo", "Caindo": "Subindo"}
agr["acerto"] = (agr.estado == agr.verdade_estado).astype(float)
agr["dir_errada"] = (agr.estado == agr.verdade_estado.map(oposto)).astype(float)
agr["erro_pp"] = (np.expm1(agr.est) - np.expm1(agr.verdade_log)).abs() * 100
print(len(agr), "linhas;", REPS_AGR, "repetições por célula")

contagem ok 404 s


proporção ok 501 s
27000 linhas; 250 repetições por célula


In [16]:
def tab_agr(col):
    g = agr.groupby(["tipo", "cenario", "D", "modo"])[col]
    m, n = g.mean(), g.size()
    ep = np.sqrt(m * (1 - m) / n)
    txt = (100 * m).round(0).astype(int).astype(str) + " ± " + (100 * ep).round(0).astype(int).astype(str)
    return txt.unstack("modo")


print("ACERTO do estado de 'Todos' (% das repetições; ± erro de Monte Carlo):")
display(tab_agr("acerto"))
print("DIREÇÃO ERRADA (Subindo no lugar de Caindo ou o inverso, %):")
display(tab_agr("dir_errada"))

vies = agr.assign(est_pct=100 * np.expm1(agr.est), verdade_pct=100 * np.expm1(agr.verdade_log))
v = vies.groupby(["tipo", "cenario", "D", "modo"]).agg(m=("est_pct", "mean"), ep=("est_pct", lambda x: x.std() / np.sqrt(len(x))), v=("verdade_pct", "first"))
print("VARIAÇÃO ESTIMADA do conjunto (média em %, ± erro de Monte Carlo) contra a verdade do conjunto (coluna 'verdade'):")
tab = (v.m.round(1).astype(str) + " ± " + v.ep.round(1).astype(str)).unstack("modo")
tab.insert(0, "verdade", v.v.groupby(level=[0, 1, 2]).first().round(1))
display(tab)

ACERTO do estado de 'Todos' (% das repetições; ± erro de Monte Carlo):


modo                  mediana_tendencias meta_efeitos_aleatorios normalizada normalizada_peso_igual soma_bruta
tipo      cenario D                                                                                           
contagem  A       1               95 ± 1                  95 ± 1      83 ± 2                 94 ± 2     95 ± 1
                  10              94 ± 1                  92 ± 2      80 ± 3                 93 ± 2     94 ± 2
                  30              95 ± 1                  94 ± 2      79 ± 3                 93 ± 2     92 ± 2
                  100             94 ± 2                  92 ± 2      80 ± 3                 93 ± 2     90 ± 2
          B       1               82 ± 2                  94 ± 2      94 ± 1                 96 ± 1     12 ± 2
                  10              87 ± 2                  95 ± 1      95 ± 1                 97 ± 1      7 ± 2
                  30              82 ± 2                  92 ± 2      94 ± 2                 95 ± 1      5 ± 1
                  100             79 ± 3                  94 ± 2      94 ± 1                 96 ± 1      3 ± 1
          C       1               79 ± 3                  66 ± 3      68 ± 3                 93 ± 2      9 ± 2
                  10              79 ± 3                  68 ± 3      67 ± 3                 93 ± 2      1 ± 1
                  30              83 ± 2                  70 ± 3      70 ± 3                 94 ± 2      1 ± 1
                  100             78 ± 3                  66 ± 3      68 ± 3                 90 ± 2      1 ± 1
proporção A       1               94 ± 2                  90 ± 2      75 ± 3                    NaN     90 ± 2
                  10              95 ± 1                  94 ± 1      76 ± 3                    NaN     74 ± 3
                  30              98 ± 1                  93 ± 2      79 ± 3                    NaN     58 ± 3
                  100             96 ± 1                  92 ± 2      80 ± 3                    NaN     48 ± 3
          B       1               96 ± 1                 100 ± 0      78 ± 3                    NaN     36 ± 3
                  10              98 ± 1                 100 ± 0      76 ± 3                    NaN     24 ± 3
                  30              96 ± 1                 100 ± 0      77 ± 3                    NaN     22 ± 3
                  100             96 ± 1                 100 ± 0      66 ± 3                    NaN     11 ± 2
          C       1               97 ± 1                  95 ± 1      56 ± 3                    NaN     27 ± 3
                  10              93 ± 2                  82 ± 2      60 ± 3                    NaN      6 ± 2
                  30              97 ± 1                  85 ± 2      52 ± 3                    NaN      0 ± 0
                  100             97 ± 1                  81 ± 2      48 ± 3                    NaN      0 ± 0

DIREÇÃO ERRADA (Subindo no lugar de Caindo ou o inverso, %):


modo                  mediana_tendencias meta_efeitos_aleatorios normalizada normalizada_peso_igual soma_bruta
tipo      cenario D                                                                                           
contagem  A       1                0 ± 0                   0 ± 0       0 ± 0                  0 ± 0      0 ± 0
                  10               0 ± 0                   0 ± 0       0 ± 0                  0 ± 0      0 ± 0
                  30               0 ± 0                   0 ± 0       0 ± 0                  0 ± 0      0 ± 0
                  100              0 ± 0                   0 ± 0       0 ± 0                  0 ± 0      0 ± 0
          B       1                0 ± 0                   0 ± 0       0 ± 0                  0 ± 0      0 ± 0
                  10               0 ± 0                   0 ± 0       0 ± 0                  0 ± 0      0 ± 0
                  30               0 ± 0                   0 ± 0       0 ± 0                  0 ± 0      1 ± 1
                  100              0 ± 0                   0 ± 0       0 ± 0                  0 ± 0      4 ± 1
          C       1                0 ± 0                   0 ± 0       0 ± 0                  0 ± 0      0 ± 0
                  10               0 ± 0                   0 ± 0       0 ± 0                  0 ± 0      2 ± 1
                  30               0 ± 0                   0 ± 0       0 ± 0                  0 ± 0      4 ± 1
                  100              0 ± 0                   0 ± 0       0 ± 0                  0 ± 0      4 ± 1
proporção A       1                0 ± 0                   0 ± 0       0 ± 0                    NaN      0 ± 0
                  10               0 ± 0                   0 ± 0       0 ± 0                    NaN      0 ± 0
                  30               0 ± 0                   0 ± 0       0 ± 0                    NaN      0 ± 0
                  100              0 ± 0                   0 ± 0       0 ± 0                    NaN      0 ± 0
          B       1                0 ± 0                   0 ± 0       0 ± 0                    NaN      0 ± 0
                  10               0 ± 0                   0 ± 0       0 ± 0                    NaN      0 ± 0
                  30               0 ± 0                   0 ± 0       0 ± 0                    NaN      0 ± 0
                  100              0 ± 0                   0 ± 0       0 ± 0                    NaN      1 ± 1
          C       1                0 ± 0                   0 ± 0       0 ± 0                    NaN      0 ± 0
                  10               0 ± 0                   0 ± 0       0 ± 0                    NaN      4 ± 1
                  30               0 ± 0                   0 ± 0       0 ± 0                    NaN     16 ± 2
                  100              0 ± 0                   0 ± 0       0 ± 0                    NaN     29 ± 3

VARIAÇÃO ESTIMADA do conjunto (média em %, ± erro de Monte Carlo) contra a verdade do conjunto (coluna 'verdade'):


modo                   verdade mediana_tendencias meta_efeitos_aleatorios normalizada normalizada_peso_igual   soma_bruta
tipo      cenario D                                                                                                      
contagem  A       1     -1.400         -0.8 ± 0.3              -2.4 ± 0.2  -3.6 ± 0.2             -1.2 ± 0.2   -1.3 ± 1.0
                  10    -1.400         -1.2 ± 0.3              -2.8 ± 0.2  -3.9 ± 0.2             -1.6 ± 0.2  -10.3 ± 1.3
                  30    -1.400         -0.5 ± 0.3              -2.3 ± 0.2  -3.4 ± 0.2             -1.1 ± 0.2  -11.3 ± 2.0
                  100   -1.400         -0.7 ± 0.3              -2.4 ± 0.2  -3.5 ± 0.2             -1.2 ± 0.2   -9.2 ± 3.0
          B       1     14.400         14.2 ± 0.3              13.7 ± 0.3  13.1 ± 0.2             14.0 ± 0.3   15.7 ± 1.2
                  10    14.400         14.8 ± 0.3              14.3 ± 0.3  13.6 ± 0.3             14.7 ± 0.3   13.3 ± 1.7
                  30    14.400         14.5 ± 0.4              13.7 ± 0.3  13.0 ± 0.3             14.1 ± 0.3   16.0 ± 2.8
                  100   14.400         14.2 ± 0.3              13.5 ± 0.3  12.8 ± 0.3             13.9 ± 0.3   19.2 ± 4.0
          C       1     12.800         14.3 ± 0.3              11.7 ± 0.3   8.9 ± 0.2             12.7 ± 0.3   12.0 ± 1.2
                  10    12.800         14.1 ± 0.3              11.5 ± 0.2   8.7 ± 0.2             12.5 ± 0.2    1.4 ± 1.4
                  30    12.800         14.4 ± 0.3              12.0 ± 0.3   9.1 ± 0.2             12.9 ± 0.2   -3.1 ± 2.0
                  100   12.800         13.8 ± 0.3              11.5 ± 0.3   8.8 ± 0.3             12.4 ± 0.3    1.5 ± 3.3
proporção A       1     -1.400         -0.8 ± 0.2              -2.2 ± 0.2  -4.0 ± 0.4                    NaN   -3.0 ± 0.5
                  10    -1.400         -0.8 ± 0.2              -2.2 ± 0.2  -3.6 ± 0.4                    NaN   -8.8 ± 0.6
                  30    -1.400         -0.7 ± 0.2              -2.3 ± 0.2  -3.8 ± 0.4                    NaN  -12.6 ± 0.6
                  100   -1.400         -0.6 ± 0.2              -2.3 ± 0.2  -3.6 ± 0.5                    NaN  -17.0 ± 0.6
          B       1     14.400         14.2 ± 0.2              13.5 ± 0.2  14.2 ± 0.4                    NaN   13.3 ± 0.5
                  10    14.400         14.0 ± 0.2              13.2 ± 0.2  14.5 ± 0.5                    NaN   11.1 ± 0.6
                  30    14.400         14.3 ± 0.2              13.1 ± 0.2  16.0 ± 0.5                    NaN   10.0 ± 0.6
                  100   14.400         13.7 ± 0.2              12.8 ± 0.2  15.3 ± 0.6                    NaN    6.0 ± 0.6
          C       1     12.800         14.1 ± 0.2              12.0 ± 0.2  10.3 ± 0.4                    NaN   10.2 ± 0.5
                  10    12.800         13.6 ± 0.2              11.5 ± 0.2  11.2 ± 0.5                    NaN    0.8 ± 0.7
                  30    12.800         14.2 ± 0.2              11.8 ± 0.2  10.7 ± 0.4                    NaN   -6.2 ± 0.6
                  100   12.800         14.1 ± 0.2              11.8 ± 0.2  12.1 ± 0.5                    NaN  -10.4 ± 0.7

In [17]:
resumo_agr = agr.groupby(["tipo", "modo"]).agg(
    acerto=("acerto", "mean"), dir_errada=("dir_errada", "mean"), erro_pp=("erro_pp", "mean")
)
resumo_agr["acerto_ep"] = np.sqrt(resumo_agr.acerto * (1 - resumo_agr.acerto) / agr.groupby(["tipo", "modo"]).size())
print("Resumo geral (todas as células): erro absoluto médio da variação do conjunto em pp:")
(resumo_agr * [100, 100, 1, 100]).round(1)

Resumo geral (todas as células): erro absoluto médio da variação do conjunto em pp:


acerto  dir_errada  erro_pp  acerto_ep
tipo      modo                                                           
contagem  mediana_tendencias       85.700       0.000    4.000      0.600
          meta_efeitos_aleatorios  84.900       0.000    3.200      0.700
          normalizada              81.000       0.000    3.800      0.700
          normalizada_peso_igual   93.900       0.000    3.100      0.400
          soma_bruta               34.100       1.300   26.100      0.900
proporção mediana_tendencias       96.000       0.000    2.900      0.400
          meta_efeitos_aleatorios  92.600       0.000    2.400      0.500
          normalizada              68.500       0.000    5.900      0.800
          soma_bruta               32.900       4.100   12.000      0.900

## 8. Experimento 4: tendência não linear (campanha, sazonalidade, degrau)

O modelo da fase 1 supõe tendência log-linear. Aqui a curva verdadeira não é uma reta. Para cada curva guardamos duas "verdades":
a **variação ponta a ponta** (valor no fim contra o início) e a **inclinação da melhor reta** sobre a curva sem ruído (o que o
método linear é capaz de enxergar). Cenário `base` de ruído. Também se avalia um sinalizador simples de não linearidade (termo
quadrático significativo a 5%) como alerta possível na interface.

In [18]:
CURVAS = {
    "reta +15% (referência)": lambda t: np.log(1.15) * t,
    "campanha no início (pico ×2 em t=0,2)": lambda t: np.log(2.0) * np.exp(-0.5 * ((t - 0.2) / 0.06) ** 2),
    "campanha no meio (pico ×2 em t=0,5)": lambda t: np.log(2.0) * np.exp(-0.5 * ((t - 0.5) / 0.06) ** 2),
    "campanha no fim (pico ×2 em t=0,85)": lambda t: np.log(2.0) * np.exp(-0.5 * ((t - 0.85) / 0.06) ** 2),
    "sazonalidade (ciclo de 30 dias, ±40%)": lambda t: np.log(1.4) * np.sin(2 * np.pi * 3 * t),
    "degrau permanente +30% em t=0,6": lambda t: np.where(t >= 0.6, np.log(1.3), 0.0),
}

fig, ax = plt.subplots(figsize=(8, 3.4))
grade = np.linspace(0, 1, 200)
for nome, f in CURVAS.items():
    ax.plot(grade * PERIODO_DIAS, np.expm1(f(grade)) * 100, label=nome)
ax.set_xlabel("dia do período")
ax.set_ylabel("variação vs. início (%)")
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()


def flag_nao_linear(df):
    t = df["t"].to_numpy() - 0.5
    X = np.column_stack([np.ones(len(t)), t, t**2])
    b, cov, gl = ols_rapido(np.log(df["y"].to_numpy()), X)
    return 2 * stats.t.sf(abs(b[2] / np.sqrt(cov[2, 2])), gl) < 0.05


rng = np.random.default_rng(SEED + 41)
linhas = []
for nome, f in CURVAS.items():
    ponta = np.expm1(f(np.array(1.0)) - f(np.array(0.0)))
    reta = np.expm1(np.polyfit(grade, f(grade), 1)[0])
    for n in [40, 100, 250]:
        est_ = []
        for _ in range(N_NL):
            df = gerar_perfil(rng, n, 0.0, "base", curva=f)
            r = m_ols_controles(df)
            est_.append((n, df["dia"].nunique(), *r, flag_nao_linear(df)))
        d = pd.DataFrame(est_, columns=["npub", "ndias", "est", "lo", "hi", "flag"])
        d["estado"] = classificar(d)
        vc = d.estado.value_counts(normalize=True)
        linhas.append(
            {
                "curva": nome, "publicações": n,
                "ponta a ponta %": 100 * ponta, "melhor reta %": 100 * reta,
                "Subindo %": 100 * vc.get("Subindo", 0), "Caindo %": 100 * vc.get("Caindo", 0),
                "Estável %": 100 * vc.get("Estável", 0), "alerta não linear %": 100 * d.flag.mean(),
                "variação média estimada %": 100 * np.expm1(d.est.mean()),
            }
        )
nl = pd.DataFrame(linhas).set_index(["curva", "publicações"])
print(f"{N_NL} séries por linha; erro de MC dos percentuais de estado ≤ {100 * 0.5 / np.sqrt(N_NL):.1f} pp")
nl.round(1)

600 séries por linha; erro de MC dos percentuais de estado ≤ 2.0 pp


ponta a ponta %  melhor reta %  Subindo %  Caindo %  Estável %  alerta não linear %  variação média estimada %
curva                                 publicações                                                                                                                
reta +15% (referência)                40                    15.000         15.000      6.000     0.800     93.200                3.700                     13.400
                                      100                   15.000         15.000     11.500     0.000     88.500                5.200                     16.000
                                      250                   15.000         15.000     20.000     0.200     79.800                4.700                     13.400
campanha no início (pico ×2 em t=0,2) 40                    -0.300        -30.900      0.200    22.000     77.800                6.700                    -30.900
                                      100                   -0.300        -30.900      0.000    47.800     52.200                8.200                    -31.300
                                      250                   -0.300        -30.900      0.000    81.500     18.500                9.700                    -31.100
campanha no meio (pico ×2 em t=0,5)   40                     0.000         -0.000      2.300     3.500     94.200               25.300                     -2.000
                                      100                    0.000         -0.000      2.700     2.500     94.800               51.200                      0.200
                                      250                    0.000         -0.000      3.300     2.200     94.500               84.000                      0.500
campanha no fim (pico ×2 em t=0,85)   40                     3.100         53.400     32.800     0.000     67.200               10.000                     57.600
                                      100                    3.100         53.400     62.000     0.000     38.000               22.300                     57.300
                                      250                    3.100         53.400     91.800     0.000      8.200               46.300                     55.300
sazonalidade (ciclo de 30 dias, ±40%) 40                    -0.000        -19.000      0.300     9.700     90.000                6.500                    -19.700
                                      100                   -0.000        -19.000      0.200    18.700     81.200                6.500                    -18.700
                                      250                   -0.000        -19.000      0.300    43.700     56.000                9.700                    -19.100
degrau permanente +30% em t=0,6       40                    30.000         45.600     22.700     0.000     77.300                5.200                     46.200
                                      100                   30.000         45.600     52.800     0.000     47.200                6.700                     46.300
                                      250                   30.000         45.600     89.800     0.000     10.200               15.800                     46.400

## 9. Experimento 5 (opcional): teste de fumaça com a Silver local do piloto

**Não é avaliação e nunca escolhe método.** Só confirma que o código roda nas colunas reais e mostra a distribuição de estados que
o contrato daria. Leitura somente-leitura da Silver local, sem `DeltaRepository`. A proporção usada é um **proxy** (fração de
comentários com 30 caracteres ou mais por reel), porque a Silver não tem sentimento.

In [19]:
def achar_silver():
    candidatos = [p / "data" / "silver" for p in [Path.cwd(), *Path.cwd().parents]]
    candidatos.append(Path(r"C:\Projetos\Tecnicas-de-NLP-em-dados-do-Instagram\data\silver"))
    for c in candidatos:
        if (c / "reels_clean").exists() and (c / "comments_clean").exists():
            return c
    return None


silver = achar_silver()
if silver is None:
    print("Silver local não encontrada; célula ignorada.")
else:
    try:
        from deltalake import DeltaTable

        reels = DeltaTable(str(silver / "reels_clean")).to_pandas()
        coment = DeltaTable(str(silver / "comments_clean")).to_pandas()
        reels["_dt"] = pd.to_datetime(reels["data_hora"], errors="coerce")
        reels = reels.dropna(subset=["_dt"])
        longo = coment.assign(longo=(coment["comprimento texto"] >= 30).astype(int))
        por_reel = longo.groupby("id_reel").agg(k=("longo", "sum"), n=("longo", "size")).reset_index()
        reels = reels.merge(por_reel, left_on="id", right_on="id_reel", how="left")
        saida = []
        for perfil, g in reels.groupby("ownerUsername"):
            dias = (g["_dt"] - g["_dt"].min()).dt.days.to_numpy()
            t = dias / 90
            base = pd.DataFrame({"dia": dias, "t": t, "formato": 0})
            dc_ = base.assign(y=g["Total de Engajamento"].clip(lower=1).to_numpy())
            gp = g.dropna(subset=["n"])
            dp_ = pd.DataFrame(
                {"dia": dias[g["n"].notna().to_numpy()], "t": t[g["n"].notna().to_numpy()], "formato": 0,
                 "n": gp["n"].to_numpy(float), "k": gp["k"].to_numpy(float)}
            )
            for nome, fn, df in [("contagem ols_controles", m_ols_controles, dc_), (f"proporção {PROP_CANDIDATO}", METODOS_PROP[PROP_CANDIDATO], dp_)]:
                try:
                    r = fn(df) if len(df) > 4 else None
                except Exception:
                    r = None
                saida.append({"perfil": perfil, "indicador": nome, "pubs": len(df), "estado": estado_unico(len(df), df["dia"].nunique(), r)})
        real = pd.DataFrame(saida)
        print(real.pivot_table(index="indicador", columns="estado", values="perfil", aggfunc="count", fill_value=0))
        print("mediana de publicações por perfil:", real.groupby("indicador").pubs.median().to_dict())
    except Exception as exc:  # fumaça opcional: nunca derruba o notebook
        print("Teste de fumaça ignorado:", repr(exc))

estado                  Caindo  Dados insuficientes  Estável
indicador                                                   
contagem ols_controles       1                   24        0
proporção ols_ref            0                   24        1
mediana de publicações por perfil: {'contagem ols_controles': 17.0, 'proporção ols_ref': 17.0}


In [20]:
print(f"Tempo total de execução até aqui: {(time.time() - T0) / 60:.1f} min")

Tempo total de execução até aqui: 8.5 min


## 10. Leitura dos resultados (fase 2a, preliminar)

200 repetições × 26 perfis por célula no experimento 1 (5.200 séries por célula de cenário e tamanho), 1.000 séries por célula de
efeito no experimento 2a, 20.000 séries por indicador no 2b, 250 repetições por célula no experimento 3 e 600 séries por linha no
experimento 4. O erro de Monte Carlo informado ao lado dos números é de ~0,1 a 0,5 pp nos totais do experimento 1 e de até ~3 pp
nas células do experimento 3; diferenças menores que 2 erros-padrão não são conclusivas.

**Checagens de sanidade (com asserção)**: as implementações próprias batem com `statsmodels`; em série sem ruído todos os métodos
recuperam a variação conhecida; e, no caso em que o modelo vale (binomial pura, sem sobredispersão nem post viral), o
quase-binomial e a meta-regressão têm cobertura de 95,5% e 97,0%. Ou seja, o desempenho ruim de alguns candidatos abaixo não é bug
de código: vem da heterogeneidade do gerador (posts com muitos comentários pesam demais). A reprodução da fase 1 para contagem
manteve o OLS com controles calibrado (falso alarme 4,8 a 5,6%, cobertura 94,6 a 95,2%).

### 10.1 Indicadores em proporção (experimento 1)

- **Candidato recomendado: `ols_ref`**, o OLS de `log((k+0,5)/(n+1))` em tempo + formato, sem peso, ou seja, o método da fase 1
  aplicado à proporção com correção de continuidade. Totais (4 cenários × 3 tamanhos): falso alarme 4,9 ± 0,1%, poder 21,5 ± 0,2%,
  direção errada 0,4%, cobertura do IC 94,7 ± 0,1%. Pior caso entre cenários: falso alarme 5,2% (`mix`) e cobertura 94,5%
  (`mix`). É o único que passa na regra de calibração (falso alarme ≤ 6%, cobertura ≥ 93% em todos os cenários).
- **`meta_regressao_logit`** (logit empírico, efeitos aleatórios por post, Knapp–Hartung) é praticamente equivalente (4,6% /
  19,9% / cobertura 95,3%), mas passa de 6% de falso alarme em `poucos` comentários (6,6%). Fica como alternativa se o dado real
  tiver muitas publicações com poucos comentários.
- **`cap_robusto`** (peso limitado + erro-padrão robusto) tem 2 pp a mais de poder (23,7%), mas calibração no limite (falso alarme
  7,7% e cobertura 92,3% no pior cenário) e maior erro de magnitude. Não recomendado.
- **A regressão logística ponderada pelo número de comentários, que era a primeira ideia, não serve**: o `quase_binomial` tem 25,8%
  de falso alarme, cobertura de 74,2% e 5,9% de direção errada (no `viral`: 37,9% de falso alarme e cobertura de 61,5%). A
  correção de sobredispersão não resolve, porque os posts com muitos comentários têm peso desproporcional e carregam o ruído entre
  posts. O erro-padrão robusto melhora (13,8% de falso alarme), mas continua descalibrado. A versão sem correção
  (`binomial_ingenuo`) tem 49,5% de falso alarme.
- **Controlar o formato continua pesando**: no `mix`, o quase-binomial sem controle tem 33,8% de falso alarme contra 23,5% com
  controle. O `ols_ref` com controle fica em 5,2%.
- **Proporção por janela com intervalo de Wilson** (`janelas_wilson`) não serve (55,1% de falso alarme, cobertura 45,0%): o
  Wilson supõe comentários independentes, e a sobredispersão entre posts o torna muito confiante; também não controla o formato.
- **Poder** do `ols_ref`: 8,6% com 20 publicações, 19,3% com 60 e 36,8% com 150. O achado da fase 1 se repete: baixo poder, a
  maioria das setas será "Estável".

### 10.2 Poder por tamanho de efeito e sensibilidade a limiares (experimento 2)

- **Poder por efeito** (cenário `base`). Com 100 publicações: queda de 30% é detectada em 49% (contagem) e 56% (proporção); alta
  de 30% em 29% e 53%. Com 250: queda de 30% em 84% e 89%, alta de 30% em 59% e 82%. Efeitos de ±10% ficam em 12 a 24% mesmo com
  250 publicações. A assimetria queda/alta aparece porque -30% e +30% são variações diferentes em escala log.
- **Efeito mínimo detectável a 80% (EMD80)**: com 250 publicações, queda de 28,8% e alta de 42,9% (contagem); queda de 26,5% e
  alta de 29,3% (proporção). Com 100 publicações, só ~43% (contagem, queda) e 43 a 48% (proporção). Com 50 publicações ou
  menos, nenhum atinge 80% até ±50%. Em termos de produto: a seta só captura variações grandes; "Estável" é a regra.
- **`MIN_EFEITO` quase não importa**: de 0% a 15% os resultados são idênticos, e em 20% o poder da proporção cai só 0,5 pp. A
  exigência de que o IC exclua zero já impõe uma variação estimada grande. Mantê-lo em 5% tem função de proteção contra erros de
  especificação, não de calibração estatística. Faixa a testar no *backtest*: 2% a 10%.
- **Mínimos de publicações e de dias distintos** (com publicações heterogêneas entre perfis, mediana 30): `(10, 8)` deixa 10,8%
  (contagem) e 9,7% (proporção) em "Dados insuficientes"; `(20, 15)` deixa ~32%, com falso alarme de 3,1% (contra 4,2% e 4,1%) e
  poder entre as séries classificadas subindo de 13,2% para 15,5% (contagem) e de 19,4% para 22,7% (proporção); `(50, 30)` deixa
  73% sem seta. O falso alarme já está no nominal com os mínimos atuais, então subir os mínimos compra pouco em calibração e custa
  cobertura de perfis. Faixa a testar no *backtest*: publicações 10 a 20, dias distintos 8 a 15.

### 10.3 Agregação da visão "Todos" (experimento 3)

Verdade do conjunto = média dos efeitos com peso igual por perfil (decisão de produto: "Todos" representa o grupo).

- **Soma bruta não serve**: acerto de 34,1% (contagem) e 32,9% (proporção). Quando os outros 25 perfis sobem e o dominante fica
  estável (B), acerta 3 a 12% (contagem). Com dominância em volume de comentários (proporção, cenário C), dá **direção errada** em
  29 ± 3% das repetições com D = 100 e estima -10,4% quando a verdade é +12,8%. A hipótese da fase 1 (dominância forte distorce a
  soma) agora se confirma com D de 30 a 100.
- **Normalizar por perfil não basta em contagem**: o perfil que publica mais ainda pesa mais (acerto 81,0%, viés de -3,5% contra
  verdade de -1,4% no cenário A; 8,8% contra 12,8% no C). Pesar cada post por 1/n do perfil (`normalizada_peso_igual`) corrige: acerto
  93,9 ± 0,4%, erro de magnitude de 3,1 pp, nenhuma direção errada.
- **Mediana das tendências individuais** (IC por *bootstrap* de perfis): nunca deu direção errada; acerto 85,7 ± 0,6% em contagem
  e 96,0 ± 0,4% em proporção, com viés pequeno. Menos poder que a normalizada com peso igual em contagem (78 a 87% nos cenários B e C
  contra 90 a 97%). A meta-análise de efeitos aleatórios (92,6% em proporção, 84,9% em contagem) pende para o perfil mais preciso,
  que pode ser justamente o dominante (cenário C: 66 a 70% em contagem).
- **Regra provisória**: tendência do conjunto = **cada perfil pesa igual**; implementada como mediana das tendências individuais
  (única opção boa nos dois tipos de indicador, sem direção errada), com a normalizada com peso igual por perfil como alternativa de
  mais poder para contagem. Nunca soma bruta. Pendente do *backtest*.

### 10.4 Tendência não linear (experimento 4)

O modelo linear **vê a melhor reta, não a história**. Com 250 publicações e sem variação ponta a ponta (variação entre início e
fim ≈ 0): campanha no início do período produz "Caindo" em 81,5% das séries; campanha no fim produz "Subindo" em 91,8% (variação
estimada +55% contra +3% ponta a ponta); sazonalidade de 30 dias produz "Caindo" em 43,7%; campanha no meio fica "Estável" em
94,5% (a reta não vê). Um degrau permanente de +30% é estimado em +46% (a reta superestima o efeito ponta a ponta) e vira "Subindo"
em 89,8%. O sinalizador de não linearidade (termo quadrático) só funciona para a campanha no meio (84% com 250 publicações); para
campanha no início ou no fim e sazonalidade ele dispara em 10% a 46%, ou seja, **não protege** contra a leitura enganosa.

### 10.5 Teste de fumaça com o dado real do piloto

24 de 25 perfis ficaram "Dados insuficientes" nos dois indicadores (mediana de 17 publicações por perfil; a proporção usada é um
proxy). O código roda nas colunas reais; **nada a concluir sobre os métodos**.

## 11. O que a simulação sustenta, o que não sustenta, limites e próximos passos

**Sustenta**

- Para proporções, o OLS do log da proporção corrigida, com o formato como controle, é calibrado nos quatro cenários e entra no
  mesmo contrato da fase 1. A regressão logística ponderada por comentários e a proporção por janela com Wilson **não** devem ser
  usadas como seta.
- Poder por efeito: só variações da ordem de ±30% são detectadas com frequência razoável (250 publicações). `MIN_EFEITO` entre 2% e
  10% dá o mesmo resultado; os mínimos de publicações e dias distintos afetam sobretudo a cobertura de perfis.
- Na visão "Todos", a soma bruta é inaceitável; a regra de peso igual por perfil (mediana das tendências) é a recomendação
  provisória.
- Tendência linear sobre curva não linear induz seta enganosa; a interface precisa evitar a leitura de "história" a partir da seta.

**Não sustenta**

- Que o `ols_ref` seja o melhor método possível para proporções (só o melhor entre os testados sob estas premissas; a
  diferença para `meta_regressao_logit` é pequena e depende do cenário).
- Qualquer escolha final de método ou de limiar: a decisão depende do *backtest* com a Coleta de 90 dias (#256).
- Que a regra de agregação valha com mais perfis dominantes ou com tendências heterogêneas entre perfis (o experimento tem um
  dominante e efeitos homogêneos nos outros 25).

**Limites do gerador**

- Ruído log-normal homocedástico (contagem); proporção com heterogeneidade logit-normal entre posts, número de comentários
  log-normal independente da proporção e do tempo. No dado real, número de comentários e sentimento podem se correlacionar.
- Tendência log-linear (contagem) ou logit-linear (proporção) e efeito constante, salvo no experimento 4.
- Dias de publicação sorteados com pesos gama; sem dependência temporal entre posts (autocorrelação ignorada).
- Perfis independentes; no dado real há eventos comuns (eleição, feriado) que movem todos juntos.

**Próximos passos**

1. *Backtest* com a Coleta de 90 dias (#256): estimar numa janela e testar na seguinte, para `ols_controles` (contagem),
   `ols_ref` e `meta_regressao_logit` (proporção) e as regras de agregação.
2. Testar no *backtest* `MIN_EFEITO` de 2% a 10%, mínimo de publicações de 10 a 20 e de dias distintos de 8 a 15.
3. Decidir como a interface comunica não linearidade (por exemplo, mostrar a série junto da seta) e se a tendência por sub-período
   (filtro de período) substitui o sinalizador quadrático.
4. Avaliar tendência com autocorrelação (efeitos de evento comum) antes de fechar o ADR 0040.